<a href="https://colab.research.google.com/github/AcSsalazar/the-color-of-emotions/blob/main/Notebooks/5-model-training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CNN VS FINE-TUNING VS RANDOM FOREST


Implementando Pytorch/Torchvision, podemos comparar o desempenño de um modelo CNN pré-treinado, un modelo fine-tuning y unmodelo Random Forest para la tarea de clasificacion de imagenes.

El imput para probar nuestros modelos sera un archivo de audio nuevo y grabado por nosotros mismos, este clip primero sera filtrado, recortado y convertido a un espectrograma, los parametros de dicha transformación deberan ajustarse al formato de entrada al que hemos configurado como nuevos inputs para nuestros modelos.

Para ello sera destinada una web app, donde el usuario pueda subir su clip de audio, y el sistema se encargara de procesarlo y mostrar los resultados de las predicciones de cada modelo.

In [3]:
# importando librerias necesarias

import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from torch.cuda.amp import GradScaler, autocast # Para eficiencia en memoria
from google.colab import drive


In [16]:
# Conectar a Google Drive

drive.mount('/content/drive')
ROOT_DIR = '/content/drive/MyDrive/ravdess_images'  # Cambia esto a tu ruta

# Configuración del dispositivo (GPU si está disponible, de lo contrario CPU)
# Se recomienda usar la GPU que ofrece Google Colab para acelerar el entrenamiento de

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Estas en modo: {device}")

# ImageFolder asume automaticamente que dentro de la carpeta en roo_dir estan las clases
# En nuestro caso, son las emociones.

def resnet_feature_selector(self, feature_type):

  # Preprocessing: Resize for ResNet, conversio a tensores, normalizar (ImageNet stats )
  data_transforms = transforms.Compose([
      #transforms.Resize((64, 64)),
      transforms.ToTensor(),
      transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalización para canales RGB necesario en ResNet
  ])

  # Define paths
  # Assuming your structure is: ./dataset/features/mfcc/calm/01.png

  TARGET_FEATURE = feature_type # Change to 'spec', 'melspect', 'delta', etc.
  FEATURE_DIR = os.path.join(ROOT_DIR, TARGET_FEATURE)

  # Use PyTorch's native ImageFolder!
  full_dataset = datasets.ImageFolder(root=FEATURE_DIR, transform=data_transforms)

  # The class map is automatically generated from folder names
  class_names = full_dataset.classes
  print(f"Emociones detectadas: {class_names}")
  print(f"Total images para la clase {TARGET_FEATURE}: {len(full_dataset)}")

  # 80-10-10 Split for Train, Validation, Test
  train_size = int(0.8 * len(full_dataset))
  val_size = int(0.1 * len(full_dataset))
  test_size = len(full_dataset) - train_size - val_size

  train_dataset, val_dataset, test_dataset = random_split(
      full_dataset, [train_size, val_size, test_size],
      generator=torch.Generator().manual_seed(42)
  )

  # Create DataLoaders
  batch_size = 32
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
  val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
  test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

  print("\n")
  print("El tamaño de los sets es:")
  print(f"Entrenamiento: {len(train_dataset)}")
  print(f"Validacion: {len(val_dataset)}")
  print(f"Test: {len(test_dataset)}")

  return train_loader, val_loader, test_loader

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Estas en modo: cuda


In [8]:
# Print root_dir content:
print(os.listdir(ROOT_DIR))

['mfcc', 'delta', 'delta2', 'spec', 'mel_spec']


In [15]:
resnet_feature_selector(ROOT_DIR, 'mel_spec')

Emociones detectadas: ['angry', 'calm', 'disgust', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Total images para la clase mel_spec: 1433


El tamaño de los sets es:
Entrenamiento: 1146
Validacion: 143
Test: 144


In [16]:
# Initialize Dataset (change 'root_dir' to your actual path)

TARGET_FEATURE = 'mel_spec' # You can change this to 'MFCC', 'spec', etc.

full_dataset = RavdessImageDataset(root_dir=ROOT_DIR, feature_type=TARGET_FEATURE, transform=transform)
num_classes = len(full_dataset.features)
print(num_classes)

# 80-10-10 Split for Train, Validation, Test
'''train_size = int(0.8 * len(full_dataset))
val_size = int(0.1 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42) # For reproducibility
)

# DataLoaders (adjust batch_size based on your GPU memory; 32 or 64 is standard)
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)'''

5


'train_size = int(0.8 * len(full_dataset))\nval_size = int(0.1 * len(full_dataset))\ntest_size = len(full_dataset) - train_size - val_size\n\ntrain_dataset, val_dataset, test_dataset = random_split(\n    full_dataset, [train_size, val_size, test_size], \n    generator=torch.Generator().manual_seed(42) # For reproducibility\n)\n\n# DataLoaders (adjust batch_size based on your GPU memory; 32 or 64 is standard)\nbatch_size = 32\ntrain_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)\nval_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)\ntest_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)'